In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://cais.gsi.go.jp/YOCHIREN/"
REPORT_LIST = urljoin(BASE_URL, "report.html")
CRAWL_DELAY = 3
UA = "research-bot/0.1 (+@example.com)"

KEYWARD = "東海・紀伊半島・四国における短期的スロースリップイベント"
KEYWARD2 = "西南日本における短期的スロースリップイベント"
KEYWARD3 = "西南日本の短期的スロースリップ活動"
KEYWARD4 = "西南日本における短期的スロースリップ- イベント"

OUTDIR = Path("downloads")
OUTDIR.mkdir(exist_ok=True)
YEAR_OK = set(range(2007, 2014))


def yymm_int(y: int, m: int) -> int:
    return y * 100 + m


def find_block_for(year: int, month: int, volumes: list[dict]):
    ym = yymm_int(year, month)
    for b in volumes:
        s_ym = yymm_int(b["start"][0], b["start"][1])
        e_ym = yymm_int(b["end"][0], b["end"][1]) if b["end"] else None
        if e_ym is None:
            if ym >= s_ym:
                return b
        else:
            if s_ym <= ym <= e_ym:
                return b
    return None


def get_volumes_data(session: requests.Session) -> list[dict]:
    r = session.get(REPORT_LIST, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    range_re = re.compile(
        r"（\s*(\d{4})年(\d{1,2})月\s*〜\s*(?:(\d{4})年(\d{1,2})月)?\s*）"
    )
    vols = []
    for a in soup.select('a[href*="report.html?id="]'):
        href = a.get("href")
        parent_text = a.parent.get_text(" ", strip=True)
        m = range_re.search(parent_text)
        if not m:
            continue
        y1, m1, y2, m2 = m.groups()
        start = (int(y1), int(m1))
        end = (int(y2), int(m2)) if y2 and m2 else None

        vols.append(
            {
                "title": a.get_text(strip=True),
                "start": start,
                "end": end,
                "href": href,
            }
        )
    return vols


def list_index_links_in_volume(
    session: requests.Session, volume_href: str, seen_index: set
) -> list[str]:
    url = urljoin(BASE_URL, volume_href)
    r = session.get(url, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")
    index_re = re.compile(r"^report/index\d+\.html$")
    range_re = re.compile(r"（\s*(\d{4})年(\d{1,2})月\s*）")

    hrefs = []
    for a in soup.find_all("a", href=index_re):
        href_raw = a.get("href")
        if href_raw in seen_pdf:
            continue

        dt = a.find_parent("dt")
        text_block = dt.get_text(" ", strip=True)
        m = range_re.search(text_block)
        if not m:
            continue

        year = int(m.group(1))
        if year not in YEAR_OK:
            continue

        hrefs.append(href_raw)
        seen_index.add(href_raw)

    return hrefs, seen_index


def download_matching_pdfs(
    session: requests.Session,
    index_href: str,
    keyword: str,
    keyword2: str,
    keyword3: str,
    keyword4: str,
):
    index_url = urljoin(BASE_URL, index_href)
    print(f"[INDEX] {index_url}")
    r = session.get(index_url, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    primary_hits = []
    secondary_hits = []
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True)
        href = a["href"]

        if not href.lower().endswith(".pdf"):
            continue

        pdf_url = urljoin(index_url, href)

        if keyword in text:
            fname = f"{text.replace(keyword, '').strip() or text}.pdf"
            primary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword2 in text:
            fname = f"{text.replace(keyword2, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword3 in text:
            fname = f"{text.replace(keyword3, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword4 in text:
            fname = f"{text.replace(keyword4, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))

    hits = primary_hits if primary_hits else secondary_hits

    for pdf_url, outpath in hits:
        try:
            print(f"[PDF]   {pdf_url}")
            resp = session.get(pdf_url, timeout=60)
            resp.encoding = resp.apparent_encoding
            resp.raise_for_status()
            outpath.write_bytes(resp.content)
            print(f"[SAVE]  {outpath}")
        except Exception as e:
            print(f"[WARN]  {pdf_url} -> {e}")
        time.sleep(CRAWL_DELAY)


seen_pdf = set()

with requests.Session() as session:
    session.headers.update({"User-Agent": UA})

    volumes = get_volumes_data(session)

    target_volume_hrefs = []
    seen_href = set()
    seen_index = set()

    for year in YEAR_OK:
        for month in range(1, 13):
            b = find_block_for(year, month, volumes)
            if not b:
                continue
            href = b["href"]
            if href not in seen_href:
                seen_href.add(href)
                target_volume_hrefs.append(href)

    for vhref in target_volume_hrefs:
        index_list, seen_index = list_index_links_in_volume(session, vhref, seen_index)

        for ihref in index_list:
            download_matching_pdfs(
                session, ihref, KEYWARD, KEYWARD2, KEYWARD3, KEYWARD4
            )
            time.sleep(CRAWL_DELAY)

[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index80.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou80/08_02.pdf
[SAVE]  downloads/（2007年11月〜2008年3月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index79.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou79/09_03.pdf
[SAVE]  downloads/（2007年5-10月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index78.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou78/08_04.pdf
[SAVE]  downloads/（2007年2月〜2007年4月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index77.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou77/7-3.pdf
[SAVE]  downloads/（2006年6月〜11月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index90.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou90/06_02.pdf
[SAVE]  downloads/（2012年11月〜2013年4月）（産総研・防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index89.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou89/06_03.pdf
[SAVE]  downloads/（

In [3]:
from datetime import datetime, time
from pathlib import Path
import re

import pandas as pd


def _years_from_session_name(name: str) -> tuple[int | None, int | None, int | None]:
    nums = [int(x) for x in re.findall(r"\d+", name)]
    years = [x for x in nums if 1900 <= x <= 2100]
    if not years:
        return None, None, None
    start_year = years[0]
    end_year = years[1] if len(years) > 1 else start_year
    start_month = next((x for x in nums[1:] if 1 <= x <= 12), 1)
    return start_year, end_year, start_month


def _infer_year(month: int, source_name: str) -> int | None:
    start_year, end_year, start_month = _years_from_session_name(source_name)
    if start_year is None:
        return None
    if end_year != start_year and start_month and month < start_month:
        return end_year
    return start_year


def timestamp_sort_key(timestamp: object, source_name: str) -> pd.Timestamp:
    text = str(timestamp).strip()
    text = text.replace("午後", "PM").replace("午前", "AM")
    start_part = text.split("-")[0]
    tod = time(12) if "PM" in start_part.upper() else time(0)

    m = re.search(r"(\d{4})/(\d{1,2})/(\d{1,2})", text)
    if m:
        y, mo, d = map(int, m.groups())
        return pd.Timestamp(datetime.combine(datetime(y, mo, d), tod))

    m = re.search(r"(\d{4})/(\d{1,2})-(\d{1,2})", text)
    if m:
        y, mo, d = map(int, m.groups())
        return pd.Timestamp(datetime.combine(datetime(y, mo, d), tod))

    m = re.search(r"(\d{1,2})/(\d{1,2})", text)
    if m:
        mo, d = map(int, m.groups())
        y = _infer_year(mo, source_name)
        if y is not None:
            return pd.Timestamp(datetime.combine(datetime(y, mo, d), tod))

    m = re.search(r"(\d{4})/(\d{1,2})", text)
    if m:
        y, mo = map(int, m.groups())
        return pd.Timestamp(datetime.combine(datetime(y, mo, 1), tod))

    return pd.NaT


path = Path("data/session")
concat_list = []
for p in path.glob("_*.csv"):
    part = pd.read_csv(p)
    part["_source_file"] = p.name
    concat_list.append(part)

df = pd.concat(concat_list, ignore_index=True)
df["_sort_timestamp"] = df.apply(
    lambda row: timestamp_sort_key(row["timestamp"], row["_source_file"]), axis=1
)
df = df.sort_values(["_sort_timestamp", "timestamp"], kind="mergesort")
df = df.drop(columns=["_sort_timestamp", "_source_file"])
df.to_csv("data/sse_catalog.csv", index=False)